In [ ]:
# Shared repository paths (no model or dataset is loaded here).
from pathlib import Path as _RepositoryPath
import os as _repository_os
import sys as _repository_sys

_repository_hint = _repository_os.environ.get("TESI_REPOSITORY_ROOT")
_repository_starts = (
    [_RepositoryPath(_repository_hint).expanduser().resolve()]
    if _repository_hint else
    ([_RepositoryPath(__file__).resolve().parent] if "__file__" in globals() else [])
    + [_RepositoryPath.cwd().resolve()]
)
_repository_root = next(
    (candidate for start in _repository_starts for candidate in (start, *start.parents)
     if (candidate / "project_paths.py").is_file()), None
)
if _repository_root is None:
    raise FileNotFoundError(
        "Start Jupyter inside the repository or set TESI_REPOSITORY_ROOT to its root."
    )
if str(_repository_root) not in _repository_sys.path:
    _repository_sys.path.insert(0, str(_repository_root))
from project_paths import PROJECT_ROOT as TESI_PROJECT_ROOT
from project_paths import BASELINES_PATH as TESI_BASELINES_PATH


In [ ]:
from pathlib import Path

# ============================================================
# CHANGE ONLY THIS
# ============================================================
EXPERT_TASK = "mnli"   # "sst" oppure "mnli"

# ============================================================
# LOCAL PATHS: CURRENT NOTEBOOK FOLDER
# ============================================================
BASE_DIR = Path(str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Baselines/EvaluationExpertMNLI'))

RESULTS_DIR = BASE_DIR / "Results"
MODELS_DIR = BASE_DIR / "Models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SAVED LOCAL EXPERTS DIRECTORY
# ============================================================
TRAINED_EXPERTS_DIR = Path(
    str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Baselines/TrainingOfUncasedModel/Models')
)

# ============================================================
# DATASET DIRECTORY
# ============================================================
DATASET_DIR = Path(
    str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Preprocessing/Dataset/ProcessedDB')
)

SST_DIR = DATASET_DIR / "SST"
MNLI_DIR = DATASET_DIR / "MNLI"

# ============================================================
# TASK CONFIGURATION
# ============================================================
TASK_CONFIG = {
    "sst": {
        "expert_name": "Expert_A",
        "expert_task": "sst",
        "cross_task": "mnli",

        "expert_num_classes": 2,
        "cross_num_classes": 3,

        "expert_model_dir": TRAINED_EXPERTS_DIR / "sst_bert_base_uncased",

        "in_domain_method": "Expert_A_SST",
        "cross_task_method": "Expert_A_on_MNLI",

        "results_file": "expert_a_metrics.json",
        "cross_model_file": "expert_a_on_mnli_classifier.pt",
    },
    "mnli": {
        "expert_name": "Expert_B",
        "expert_task": "mnli",
        "cross_task": "sst",

        "expert_num_classes": 3,
        "cross_num_classes": 2,

        "expert_model_dir": TRAINED_EXPERTS_DIR / "mnli_bert_base_uncased",

        "in_domain_method": "Expert_B_MNLI",
        "cross_task_method": "Expert_B_on_SST",

        "results_file": "expert_b_metrics.json",
        "cross_model_file": "expert_b_on_sst_classifier.pt",
    }
}

if EXPERT_TASK not in TASK_CONFIG:
    raise ValueError("EXPERT_TASK must be either 'sst' or 'mnli'")

CFG = TASK_CONFIG[EXPERT_TASK]

EXPERT_NAME = CFG["expert_name"]
EXPERT_TASK = CFG["expert_task"]
CROSS_TASK = CFG["cross_task"]

EXPERT_NUM_CLASSES = CFG["expert_num_classes"]
CROSS_NUM_CLASSES = CFG["cross_num_classes"]

IN_DOMAIN_METHOD = CFG["in_domain_method"]
CROSS_TASK_METHOD = CFG["cross_task_method"]

EXPERT_MODEL_DIR = CFG["expert_model_dir"]
EXPERT_BACKBONE_DIR = EXPERT_MODEL_DIR / "bert_backbone"
EXPERT_TOKENIZER_DIR = EXPERT_MODEL_DIR / "tokenizer"
EXPERT_CLASSIFIER_PATH = EXPERT_MODEL_DIR / "classifier.pt"
EXPERT_METADATA_PATH = EXPERT_MODEL_DIR / "model_metadata.json"

RESULTS_FILE = RESULTS_DIR / CFG["results_file"]
CROSS_MODEL_FILE = MODELS_DIR / CFG["cross_model_file"]

# ============================================================
# CHECK REQUIRED PATHS
# ============================================================
required_paths = {
    "TRAINED_EXPERTS_DIR": TRAINED_EXPERTS_DIR,
    "EXPERT_MODEL_DIR": EXPERT_MODEL_DIR,
    "EXPERT_BACKBONE_DIR": EXPERT_BACKBONE_DIR,
    "EXPERT_TOKENIZER_DIR": EXPERT_TOKENIZER_DIR,
    "EXPERT_CLASSIFIER_PATH": EXPERT_CLASSIFIER_PATH,
    "DATASET_DIR": DATASET_DIR,
    "SST_DIR": SST_DIR,
    "MNLI_DIR": MNLI_DIR,
}

for name, path in required_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} does not exist: {path}")

# ============================================================
# PRINT CONFIGURATION
# ============================================================
print("Base dir:", BASE_DIR)
print("Expert name:", EXPERT_NAME)
print("Expert task:", EXPERT_TASK)
print("Cross task:", CROSS_TASK)

print("Expert model dir:", EXPERT_MODEL_DIR)
print("Expert backbone dir:", EXPERT_BACKBONE_DIR)
print("Expert tokenizer dir:", EXPERT_TOKENIZER_DIR)
print("Expert classifier path:", EXPERT_CLASSIFIER_PATH)

print("In-domain method:", IN_DOMAIN_METHOD)
print("Cross-task method:", CROSS_TASK_METHOD)

print("Results file:", RESULTS_FILE)
print("Cross model file:", CROSS_MODEL_FILE)
# In-domain measurements only by default: no cross-task head training.
RUN_CROSS_TASK_TRAINING = False


In [ ]:
# ============================================================
# CURRENT MODE: IN-DOMAIN EVALUATION
# No training
# Use saved expert backbone + saved expert classifier
# ============================================================

CURRENT_MODE = "in_domain"

METHOD_NAME = IN_DOMAIN_METHOD
EVAL_TASK = EXPERT_TASK
NUM_CLASSES = EXPERT_NUM_CLASSES
TRAINABLE_PARAMS = 0

print("Current mode:", CURRENT_MODE)
print("Method name:", METHOD_NAME)
print("Evaluation task:", EVAL_TASK)
print("Num classes:", NUM_CLASSES)
print("Trainable params:", TRAINABLE_PARAMS)
print("Backbone path:", EXPERT_BACKBONE_DIR)
print("Classifier path:", EXPERT_CLASSIFIER_PATH)

In [ ]:
# ============================================================
# LOAD IN-DOMAIN AND CROSS-TASK DATASETS
# ============================================================

import pandas as pd

TASK_FOLDER = {
    "sst": "SST",
    "mnli": "MNLI"
}

def load_task_split(task, split):
    folder = TASK_FOLDER[task]
    task_dir = DATASET_DIR / folder

    X = pd.read_parquet(task_dir / f"X_{split}.parquet")
    y = pd.read_parquet(task_dir / f"y_{split}.parquet")

    # Convert X to Series
    if isinstance(X, pd.DataFrame):
        if "text" in X.columns:
            X = X["text"]
        elif X.shape[1] == 1:
            X = X.iloc[:, 0]
        else:
            raise ValueError(f"Unexpected X columns for {task} {split}: {list(X.columns)}")

    # Convert y to Series
    if isinstance(y, pd.DataFrame):
        if "label" in y.columns:
            y = y["label"]
        elif y.shape[1] == 1:
            y = y.iloc[:, 0]
        else:
            raise ValueError(f"Unexpected y columns for {task} {split}: {list(y.columns)}")

    X = X.astype(str).reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    return X, y


# ============================================================
# IN-DOMAIN TASK
# ============================================================

X_train_in_domain, y_train_in_domain = load_task_split(EXPERT_TASK, "train")
X_val_in_domain,   y_val_in_domain   = load_task_split(EXPERT_TASK, "val")
X_test_in_domain,  y_test_in_domain  = load_task_split(EXPERT_TASK, "test")


# ============================================================
# CROSS-TASK
# ============================================================

X_train_cross, y_train_cross = load_task_split(CROSS_TASK, "train")
X_val_cross,   y_val_cross   = load_task_split(CROSS_TASK, "val")
X_test_cross,  y_test_cross  = load_task_split(CROSS_TASK, "test")


# ============================================================
# CHECKS
# ============================================================

print("Expert task:", EXPERT_TASK)
print("Cross task:", CROSS_TASK)

print("\nIn-domain:")
print("Train:", len(X_train_in_domain))
print("Val  :", len(X_val_in_domain))
print("Test :", len(X_test_in_domain))
print("Labels:", sorted(y_train_in_domain.unique()))

print("\nCross-task:")
print("Train:", len(X_train_cross))
print("Val  :", len(X_val_cross))
print("Test :", len(X_test_cross))
print("Labels:", sorted(y_train_cross.unique()))

In [ ]:
import sys
import torch
import numpy as np
import torch.nn as nn

from transformers import AutoTokenizer, AutoModel
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ============================================================
# PARAMETERS
# ============================================================
SEED = 42
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
LR_FACTOR = 0.1
LR_PATIENCE = 2
MIN_LR = 1e-6

# ============================================================
# PATHS
# ============================================================
BASELINES_PATH = str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Baselines')

# ============================================================
# IMPORT CLASSIFIER
# ============================================================
if BASELINES_PATH not in sys.path:
    sys.path.append(BASELINES_PATH)

from classifier import Classifier

# ============================================================
# SET RANDOM SEEDS
# ============================================================
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# TOKENIZER
# Load tokenizer saved with the selected expert
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(EXPERT_TOKENIZER_DIR)

# ============================================================
# LOAD SAVED EXPERT BACKBONE
# ============================================================
bert_backbone = AutoModel.from_pretrained(EXPERT_BACKBONE_DIR)

# ============================================================
# LOAD SAVED IN-DOMAIN CLASSIFIER
# This is the classifier trained together with the expert backbone
# ============================================================
expert_classifier = Classifier(class_number=EXPERT_NUM_CLASSES)
expert_classifier.load_state_dict(
    torch.load(EXPERT_CLASSIFIER_PATH, map_location=device)
)

# ============================================================
# DEFINE NEW CROSS-TASK CLASSIFIER
# This classifier will be trained on the cross-task data
# ============================================================
cross_classifier = Classifier(class_number=CROSS_NUM_CLASSES)

# ============================================================
# MOVE MODELS TO DEVICE
# ============================================================
bert_backbone = bert_backbone.to(device)
expert_classifier = expert_classifier.to(device)
cross_classifier = cross_classifier.to(device)

# ============================================================
# FREEZE BACKBONE
# In-domain: evaluation only
# Cross-task: train only the new classifier
# ============================================================
bert_backbone.eval()

for param in bert_backbone.parameters():
    param.requires_grad = False

expert_classifier.eval()

for param in expert_classifier.parameters():
    param.requires_grad = False

cross_classifier.train()

for param in cross_classifier.parameters():
    param.requires_grad = True

# ============================================================
# COMPATIBILITY ALIASES
# ============================================================
model = bert_backbone
original_classifier = expert_classifier

# ============================================================
# PARAMETER COUNTS
# ============================================================
backbone_params = sum(p.numel() for p in bert_backbone.parameters())
expert_classifier_params = sum(p.numel() for p in expert_classifier.parameters())
cross_classifier_params = sum(p.numel() for p in cross_classifier.parameters() if p.requires_grad)

print("Loaded local expert:", EXPERT_NAME)
print("Expert task:", EXPERT_TASK)
print("Cross task:", CROSS_TASK)
print("Expert backbone dir:", EXPERT_BACKBONE_DIR)
print("Expert classifier path:", EXPERT_CLASSIFIER_PATH)
print("Expert num classes:", EXPERT_NUM_CLASSES)
print("Cross num classes:", CROSS_NUM_CLASSES)
print("Backbone frozen:", not any(p.requires_grad for p in bert_backbone.parameters()))
print("Cross classifier trainable params:", f"{cross_classifier_params:,}")
print("Device:", device)

In [ ]:
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
import torch
import numpy as np

# ============================================================
# BATCH SIZE
# ============================================================
BATCH_SIZE = 32

# ============================================================
# Utility: prepare inputs
# MNLI is sentence-pair classification
# SST is single-sentence classification
# ============================================================
def prepare_texts(task, X):
    texts = X.astype(str).tolist()

    if task == "mnli":
        premises = []
        hypotheses = []

        for t in texts:
            parts = t.split("[SEP]", 1)
            premises.append(parts[0].strip())
            hypotheses.append(parts[1].strip() if len(parts) > 1 else "")

        return premises, hypotheses

    return texts, None


def prepare_labels(y):
    return y.astype(int).values


# ============================================================
# Inspect tokenized length distribution
# ============================================================
lengths = []

splits_for_length = [
    (EXPERT_TASK, X_test_in_domain, f"{EXPERT_TASK}_test_in_domain"),
    (CROSS_TASK, X_train_cross, f"{CROSS_TASK}_train_cross"),
    (CROSS_TASK, X_val_cross, f"{CROSS_TASK}_val_cross"),
    (CROSS_TASK, X_test_cross, f"{CROSS_TASK}_test_cross"),
]

for task, X_split, split_name in splits_for_length:
    texts_a, texts_b = prepare_texts(task, X_split)

    for i in tqdm(range(len(texts_a)), desc=f"Inspecting lengths {split_name}"):
        if texts_b is None:
            enc = tokenizer(texts_a[i], truncation=False)
        else:
            enc = tokenizer(texts_a[i], texts_b[i], truncation=False)

        lengths.append(len(enc["input_ids"]))

percentiles = [90, 95, 97, 99]

print("Tokenized length percentiles:")
for p in percentiles:
    print(f"{p}% :", int(np.percentile(lengths, p)))

# Match RoutedModel: fixed truncation cap, padding to the longest item in each split.
# The percentiles above are diagnostic only; they do not change this shared cap.
MAX_LEN = 256

print("Truncation cap (same as RoutedModel):", MAX_LEN)


# ============================================================
# Tokenization utility
# ============================================================
def tokenize_split(X, y, task, split_name):
    texts_a, texts_b = prepare_texts(task, X)
    labels_array = prepare_labels(y)

    if len(texts_a) != len(labels_array):
        raise ValueError(f"Length mismatch in {split_name}: X={len(texts_a)}, y={len(labels_array)}")
    if not texts_a:
        raise ValueError(f"Cannot tokenize an empty split: {split_name}")

    # Tokenize in chunks without padding. Padding each chunk independently
    # would produce different widths and would not match RoutedModel.
    encoded_split = {"input_ids": [], "attention_mask": [], "token_type_ids": []}
    for i in tqdm(range(0, len(texts_a), 1024), desc=f"Tokenizing {split_name}"):
        batch_a = texts_a[i:i + 1024]
        batch_b = texts_b[i:i + 1024] if texts_b is not None else None
        enc = tokenizer(
            batch_a,
            text_pair=batch_b,
            padding=False,
            truncation=True,
            max_length=MAX_LEN,
            return_token_type_ids=True
        )
        encoded_split["input_ids"].extend(enc["input_ids"])
        encoded_split["attention_mask"].extend(enc["attention_mask"])
        encoded_split["token_type_ids"].extend(
            enc.get("token_type_ids", [[0] * len(ids) for ids in enc["input_ids"]])
        )

    # Pad once across the entire split, after truncation, exactly as the
    # routed tokenizer(texts, text_pair=..., padding=True, max_length=256).
    encodings = tokenizer.pad(encoded_split, padding=True, return_tensors="pt")
    input_ids = encodings["input_ids"]
    attention_mask = encodings["attention_mask"]
    token_type_ids = encodings["token_type_ids"]
    labels = torch.tensor(labels_array, dtype=torch.long)

    print(f"{split_name}: padded sequence length = {input_ids.shape[1]} (cap {MAX_LEN})")
    return TensorDataset(input_ids, attention_mask, token_type_ids, labels)


# ============================================================
# Build datasets
# ============================================================
test_dataset_in_domain = tokenize_split(
    X_test_in_domain,
    y_test_in_domain,
    EXPERT_TASK,
    f"{EXPERT_TASK}_test_in_domain"
)

train_dataset_cross = tokenize_split(
    X_train_cross,
    y_train_cross,
    CROSS_TASK,
    f"{CROSS_TASK}_train_cross"
)

val_dataset_cross = tokenize_split(
    X_val_cross,
    y_val_cross,
    CROSS_TASK,
    f"{CROSS_TASK}_val_cross"
)

test_dataset_cross = tokenize_split(
    X_test_cross,
    y_test_cross,
    CROSS_TASK,
    f"{CROSS_TASK}_test_cross"
)


# ============================================================
# Build dataloaders
# ============================================================
test_loader_in_domain = DataLoader(
    test_dataset_in_domain,
    batch_size=BATCH_SIZE,
    shuffle=False
)

train_loader_cross = DataLoader(
    train_dataset_cross,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader_cross = DataLoader(
    val_dataset_cross,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader_cross = DataLoader(
    test_dataset_cross,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("Tokenization completed successfully.")
print("MAX_LEN:", MAX_LEN)
print("In-domain test loader:", EXPERT_TASK, "| batches:", len(test_loader_in_domain))
print("Cross-task train loader:", CROSS_TASK, "| batches:", len(train_loader_cross))
print("Cross-task val loader:", CROSS_TASK, "| batches:", len(val_loader_cross))
print("Cross-task test loader:", CROSS_TASK, "| batches:", len(test_loader_cross))

In [ ]:
# ============================================================
# SAME-DOMAIN MODE
# Used before in-domain evaluation
# No training here
# ============================================================

CURRENT_MODE = "in_domain"
METHOD_NAME = IN_DOMAIN_METHOD
EVAL_TASK = EXPERT_TASK
NUM_CLASSES = EXPERT_NUM_CLASSES
# 
# ------------------------------------------------------------
# Use expert backbone + saved expert classifier
# ------------------------------------------------------------
active_classifier = expert_classifier

# ------------------------------------------------------------
# Freeze backbone and classifier
# ------------------------------------------------------------
bert_backbone.eval()
active_classifier.eval()

for param in bert_backbone.parameters():
    param.requires_grad = False

for param in active_classifier.parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# Move to device
# ------------------------------------------------------------
bert_backbone = bert_backbone.to(device)
active_classifier = active_classifier.to(device)

# ------------------------------------------------------------
# Check trainable parameters
# ------------------------------------------------------------
TRAINABLE_PARAMS = (
    sum(p.numel() for p in bert_backbone.parameters() if p.requires_grad)
    + sum(p.numel() for p in active_classifier.parameters() if p.requires_grad)
)

print("Mode: same-domain evaluation")
print("Current mode:", CURRENT_MODE)
print("Method name:", METHOD_NAME)
print("Expert task:", EXPERT_TASK)
print("Evaluation task:", EVAL_TASK)
print("Num classes:", NUM_CLASSES)
print("Trainable params:", TRAINABLE_PARAMS)
print("Backbone path:", EXPERT_BACKBONE_DIR)
print("Classifier path:", EXPERT_CLASSIFIER_PATH)

In [ ]:
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Model device:", next(model.parameters()).device)
print("MAX_LEN:", MAX_LEN)

print("Mode:", CURRENT_MODE)
print("Method:", METHOD_NAME)
print("Expert task:", EXPERT_TASK)
print("Evaluation task:", EVAL_TASK)

print("In-domain test samples:", len(X_test_in_domain))
print("In-domain test batches:", len(test_loader_in_domain))

print("Cross task:", CROSS_TASK)
print("Cross-task train samples:", len(X_train_cross))
print("Cross-task val samples:", len(X_val_cross))
print("Cross-task test samples:", len(X_test_cross))

print("Cross-task train batches:", len(train_loader_cross))
print("Cross-task val batches:", len(val_loader_cross))
print("Cross-task test batches:", len(test_loader_cross))

print("Current num labels:", model.config.num_labels)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def input_fingerprint(input_ids, attention_mask, token_type_ids):
    """Same unpadded token identity used by the existing routed JSONL exports."""
    import hashlib
    import json
    tokens = [(int(i), int(t)) for i, t, m in
              zip(input_ids, token_type_ids, attention_mask) if m]
    return hashlib.sha256(json.dumps(tokens, separators=(",", ":")).encode("utf-8")).hexdigest()


def benchmark_inference(forward_batch, dataloader, device, *, warmup_batches=10, repetitions=3):
    """Time forward only, with CPU dispatch and synchronized GPU completion."""
    import time
    import numpy as np
    import torch
    from transformers import __version__ as transformers_version
    if warmup_batches < 1 or repetitions < 2 or len(dataloader.dataset) < 1:
        raise ValueError("Latency requires nonempty data, warmup >= 1 and repetitions >= 2")
    if not isinstance(dataloader.sampler, torch.utils.data.SequentialSampler):
        raise ValueError("Latency evaluation requires the same sequential inputs for all models")
    device = torch.device(device)

    def synchronize():
        if device.type == "cuda":
            torch.cuda.synchronize(device)

    batch_times, batch_sizes, per_repeat_ms = [], [], []
    lengths = set()
    dtype = None
    with torch.inference_mode():
        warmup_iter = iter(dataloader)
        for _ in range(warmup_batches):
            try:
                batch = next(warmup_iter)
            except StopIteration:
                warmup_iter = iter(dataloader)
                batch = next(warmup_iter)
            inputs = tuple(x.to(device) for x in batch[:3])
            warmup_output = forward_batch(inputs)
            dtype = str(warmup_output.dtype)
            del warmup_output
        synchronize()
        for _ in range(repetitions):
            repeat_ms, repeat_n = 0.0, 0
            for batch in dataloader:
                # Transfers, tokenization, dataloader work and metrics are not timed.
                inputs = tuple(x.to(device) for x in batch[:3])
                synchronize()
                start = time.perf_counter()
                logits = forward_batch(inputs)
                synchronize()
                elapsed_ms = 1000.0 * (time.perf_counter() - start)
                del logits
                n = int(inputs[0].shape[0])
                lengths.add(int(inputs[0].shape[1]))
                batch_times.append(elapsed_ms)
                batch_sizes.append(n)
                repeat_ms += elapsed_ms
                repeat_n += n
            if repeat_n != len(dataloader.dataset):
                raise ValueError("Latency pass did not cover the complete evaluation split")
            per_repeat_ms.append(repeat_ms / repeat_n)
    return {
        "scope": "forward on pretokenized device-resident inputs, including classifier and router if present",
        "excluded": "tokenization, data loading, host-device transfers, loss, metrics, export, FLOPs instrumentation",
        "timer": "perf_counter with CUDA synchronization before and after each forward",
        "warmup_batches": warmup_batches, "repetitions": repetitions,
        "num_examples_per_repeat": len(dataloader.dataset),
        "batch_size": dataloader.batch_size, "observed_batch_sizes": sorted(set(batch_sizes)),
        "observed_sequence_lengths": sorted(lengths),
        "mean_ms_per_batch": float(np.mean(batch_times)),
        "std_ms_per_batch": float(np.std(batch_times, ddof=1)),
        "mean_ms_per_example": float(sum(batch_times) / sum(batch_sizes)),
        "std_ms_per_example_across_repetitions": float(np.std(per_repeat_ms, ddof=1)),
        "ms_per_example_by_repeat": per_repeat_ms,
        "batch_ms_by_repeat": [batch_times[i * len(dataloader):(i + 1) * len(dataloader)]
                              for i in range(repetitions)],
        "device": str(device),
        "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "torch_version": str(torch.__version__), "transformers_version": transformers_version,
        "dtype": dtype,
        "std_note": "Timing variability; distinct from variation between training seeds",
    }



In [ ]:
import json
import torch
import numpy as np
import torch.nn as nn

from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ============================================================
# PARAMETERS
# ============================================================
CURRENT_MODE = "in_domain"
METHOD_NAME = IN_DOMAIN_METHOD
EVAL_TASK = EXPERT_TASK
NUM_CLASSES = EXPERT_NUM_CLASSES

RESULTS_FILE = Path(RESULTS_FILE)

criterion = nn.CrossEntropyLoss()

# ============================================================
# SAME-DOMAIN SETUP
# Expert backbone + saved expert classifier, no training
# ============================================================
active_classifier = expert_classifier

bert_backbone = bert_backbone.to(device)
active_classifier = active_classifier.to(device)

bert_backbone.eval()
active_classifier.eval()

for param in bert_backbone.parameters():
    param.requires_grad = False

for param in active_classifier.parameters():
    param.requires_grad = False

TRAINABLE_PARAMS = (
    sum(p.numel() for p in bert_backbone.parameters() if p.requires_grad)
    + sum(p.numel() for p in active_classifier.parameters() if p.requires_grad)
)

# ============================================================
# INFERENCE FLOPS: same convention for every evaluation notebook
# Count executed forward operators, including padding and classifier.
# This is a mathematical operation count, not GPU instruction counting.
# ============================================================
from math import prod
from collections import defaultdict
from torch.utils.flop_counter import FlopCounterMode
from torch.utils._pytree import tree_leaves
from transformers import __version__ as transformers_version


class InferenceFlopCounter(FlopCounterMode):
    """Extend PyTorch's matrix counts and expose uncovered floating-point ops."""

    # Indexing, copies, shape changes, allocation and comparisons have no
    # floating-point arithmetic under this convention. Dropout is off in eval.
    zero_flop_ops = set("""
        alias detach view _unsafe_view reshape t transpose permute slice select
        squeeze unsqueeze expand expand_as as_strided contiguous clone copy_
        _to_copy to empty empty_like empty_strided new_empty new_empty_strided
        zeros zeros_like new_zeros ones ones_like full full_like new_full fill_
        zero_ embedding index index_select gather cat stack split split_with_sizes
        unbind masked_fill masked_fill_ where eq ne lt le gt ge argmax argmin
        max min amax amin any all isfinite isnan isinf scalar_tensor
        _local_scalar_dense lift_fresh lift_fresh_copy resolve_conj dropout
        native_dropout constant_pad_nd
    """.split())

    def __init__(self):
        if not hasattr(FlopCounterMode, "_count_flops"):
            raise RuntimeError("This FLOPs counter requires PyTorch with FlopCounterMode._count_flops support.")
        super().__init__(display=False)
        self.uncovered_ops = set()

    def _count_flops(self, packet, out, args, kwargs):
        name = str(packet).split(".")[-1]
        outputs = [x for x in tree_leaves(out)
                   if isinstance(x, torch.Tensor) and x.is_floating_point()]
        if not outputs:
            return out
        n = outputs[0].numel()
        x = args[0] if args else None

        if name in {"addmm", "baddbmm"}:
            # PyTorch's default matrix formula omits the bias addition.
            alpha, beta = kwargs.get("alpha", 1), kwargs.get("beta", 1)
            flops = 2 * n * args[1].shape[-1]
            flops += n * (int(alpha != 1) + int(beta != 0) + int(beta not in (0, 1)))
        elif name in {"add", "add_", "sub", "sub_", "rsub"}:
            flops = n * (1 + int(kwargs.get("alpha", 1) != 1))
        elif name in {"mul", "mul_", "div", "div_", "true_divide", "neg",
                      "exp", "log", "sqrt", "rsqrt", "reciprocal", "tanh", "erf", "pow"}:
            flops = n
        elif name == "gelu":
            flops = n * (9 if kwargs.get("approximate", "none") == "tanh" else 5)
        elif name in {"native_layer_norm", "layer_norm"}:
            width = prod(args[1])
            rows = x.numel() // width
            weight = args[2] if len(args) > 2 else kwargs.get("weight")
            bias = args[3] if len(args) > 3 else kwargs.get("bias")
            flops = 5 * x.numel() + 2 * rows
            flops += x.numel() * (int(weight is not None) + int(bias is not None))
        elif name in {"_softmax", "softmax"}:
            dim = args[1] if len(args) > 1 else kwargs["dim"]
            flops = 4 * x.numel() - x.numel() // x.shape[dim]
        elif name in {"sum", "mean"}:
            flops = x.numel() - n + (n if name == "mean" else 0)
        elif name in {
            "scaled_dot_product_attention", "_scaled_dot_product_attention_math",
            "_scaled_dot_product_flash_attention", "_scaled_dot_product_flash_attention_for_cpu",
            "_scaled_dot_product_efficient_attention", "_scaled_dot_product_cudnn_attention"
        }:
            # Dense, non-causal BERT attention: QK + scaling/mask/softmax + AV.
            # Applies to the operator as a whole; no extra forward is performed.
            q, k, v = args[:3]
            rows = prod(q.shape[:-1])
            scores = rows * k.shape[-2]
            mask = kwargs.get("attn_mask", kwargs.get("attn_bias"))
            if mask is None and len(args) > 3 and isinstance(args[3], torch.Tensor):
                mask = args[3]
            flops = 2 * scores * (q.shape[-1] + v.shape[-1])
            flops += 5 * scores - rows
            if isinstance(mask, torch.Tensor) and mask.is_floating_point():
                flops += scores
        elif name in self.zero_flop_ops:
            return out
        elif packet in self.flop_registry:
            # Other fused attention variants may count only their matmuls.
            if "attention" in name:
                self.uncovered_ops.add(str(packet) + " (attention overhead not covered)")
            return super()._count_flops(packet, out, args, kwargs)
        else:
            self.uncovered_ops.add(str(packet))
            return out

        self.flop_counts["Global"][packet] += int(flops)
        return out


def new_inference_cost():
    return {
        "components": defaultdict(int), "operators": defaultdict(int),
        "uncovered": set(), "num_examples": 0,
        "sequence_lengths": set(), "batch_sizes": set(),
    }


def add_inference_cost(cost, counter, component, input_ids=None):
    cost["components"][component] += int(counter.get_total_flops())
    for op, value in counter.get_flop_counts().get("Global", {}).items():
        cost["operators"][str(op)] += int(value)
    cost["uncovered"].update(counter.uncovered_ops)
    # Count examples only on the backbone pass, not again on cached features.
    if input_ids is not None:
        cost["num_examples"] += int(input_ids.shape[0])
        cost["sequence_lengths"].add(int(input_ids.shape[1]))
        cost["batch_sizes"].add(int(input_ids.shape[0]))


def finish_inference_cost(cost, num_samples):
    if num_samples <= 0 or cost["num_examples"] != num_samples:
        raise ValueError("FLOPs: backbone and classifier must cover the same test examples.")
    counted = int(sum(cost["components"].values()))
    complete = not cost["uncovered"] and counted > 0
    if not complete:
        print("Inference FLOPs incomplete; see inference_cost.coverage in the saved results.")
    return {
        "method": "pytorch_executed_forward_flops_v1",
        "scope": "test inference: backbone and classifier, inputs to logits",
        "total_flops": counted if complete else None,
        "flops_per_example": float(counted / num_samples) if complete else None,
        "num_examples": int(num_samples),
        "coverage": {
            "status": "complete_for_convention" if complete else "incomplete",
            "uncovered_operators": sorted(cost["uncovered"]),
            "counted_flops": counted,
            "counted_flops_by_component": dict(cost["components"]),
            "counted_flops_by_operator": dict(sorted(cost["operators"].items())),
        },
        "convention": {
            "multiply_accumulate_flops": 2,
            "elementwise_arithmetic_or_special_function_flops": 1,
            "gelu_flops_per_element": {"exact": 5, "tanh_approximation": 9},
            "layer_norm": "5*N + 2*rows, plus N each for affine weight and bias",
            "softmax": "4*N - rows; comparisons excluded",
            "attention": "dense QK and AV matmuls plus scaling, additive mask and softmax",
            "excluded": "training/backward, loss/metrics, preprocessing, I/O, copies, indexing, comparisons",
            "interpretation": "formula-based FLOPs of executed operators; not exact GPU kernel instructions",
        },
        "input": {
            "max_len": int(MAX_LEN), "batch_size": int(BATCH_SIZE),
            "observed_sequence_lengths": sorted(cost["sequence_lengths"]),
            "observed_batch_sizes": sorted(cost["batch_sizes"]),
            "padding": "longest_in_split", "truncation": True,
            "tokenizer": str(EXPERT_TOKENIZER_DIR),
            "dtype": str(next(bert_backbone.parameters()).dtype),
        },
        "torch_version": str(torch.__version__),
        "transformers_version": transformers_version,
    }


# ============================================================
# EVALUATION FUNCTION
# ============================================================
def evaluate_in_domain(backbone, classifier, dataloader, device, task_name, num_classes):
    backbone.eval()
    classifier.eval()

    all_preds = []
    all_labels = []
    prediction_rows = []
    total_loss = 0.0
    total_samples = 0
    inference_cost = new_inference_cost()

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Evaluating on {task_name.upper()}"):
            input_ids, attention_mask, token_type_ids, labels = [x.to(device) for x in batch]

            with InferenceFlopCounter() as backbone_counter:
                outputs = backbone(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )
            add_inference_cost(inference_cost, backbone_counter, "backbone", input_ids)

            pooled_output = outputs.pooler_output
            with InferenceFlopCounter() as classifier_counter:
                logits = classifier(pooled_output)
            add_inference_cost(inference_cost, classifier_counter, "classifier")

            loss = criterion(logits, labels)
            preds = torch.argmax(logits, dim=1)

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
            start_index = total_samples - batch_size
            for offset, (ids, mask, types, label, prediction) in enumerate(zip(
                    batch[0].tolist(), batch[1].tolist(), batch[2].tolist(),
                    batch[3].tolist(), preds.detach().cpu().tolist())):
                prediction_rows.append({
                    "example_id": f"{task_name}:test:{start_index + offset}",
                    "input_sha256": input_fingerprint(ids, mask, types),
                    "label": int(label), "prediction": int(prediction)})



    import csv
    import hashlib
    prediction_path = RESULTS_DIR / f"expert_{task_name}_predictions.csv"
    partial = prediction_path.with_suffix(".csv.partial")
    with partial.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["example_id", "input_sha256", "label", "prediction"])
        writer.writeheader()
        writer.writerows(prediction_rows)
    partial.replace(prediction_path)
    def forward_for_latency(inputs):
        output = backbone(input_ids=inputs[0], attention_mask=inputs[1], token_type_ids=inputs[2])
        return classifier(output.pooler_output)
    latency = benchmark_inference(forward_for_latency, dataloader, device)
    avg_loss = total_loss / total_samples

    accuracy = accuracy_score(all_labels, all_preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="binary" if task_name == "sst" else "macro",
        zero_division=0
    )

    cm = confusion_matrix(
        all_labels,
        all_preds,
        labels=list(range(num_classes))
    )

    return {
        "loss": float(avg_loss),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "f1_average": "binary" if task_name == "sst" else "macro",
        "confusion_matrix": cm.tolist(),
        "num_samples": int(total_samples),
        "inference_cost": finish_inference_cost(inference_cost, total_samples),
        "latency": latency,
        "predictions_csv": str(prediction_path),
        "predictions_sha256": hashlib.sha256(prediction_path.read_bytes()).hexdigest()

    }

# ============================================================
# RUN IN-DOMAIN EVALUATION
# ============================================================
in_domain_metrics = evaluate_in_domain(
    backbone=bert_backbone,
    classifier=active_classifier,
    dataloader=test_loader_in_domain,
    device=device,
    task_name=EVAL_TASK,
    num_classes=NUM_CLASSES
)

# ============================================================
# BUILD RESULT ENTRY
# ============================================================
result_entry = {
    "method": METHOD_NAME,
    "mode": CURRENT_MODE,
    "expert_task": EXPERT_TASK,
    "eval_task": EVAL_TASK,
    "trainable_params": int(TRAINABLE_PARAMS),
    "results": {
        EVAL_TASK: {
            "test_metrics": in_domain_metrics
        }
    },
    "model_path": str(EXPERT_MODEL_DIR),
    "backbone_path": str(EXPERT_BACKBONE_DIR),
    "classifier_path": str(EXPERT_CLASSIFIER_PATH)
}

# ============================================================
# LOAD EXISTING JSON IF PRESENT
# ============================================================
if RESULTS_FILE.exists():
    with open(RESULTS_FILE, "r") as f:
        results_json = json.load(f)
else:
    results_json = {"methods": []}

results_json["methods"] = [
    x for x in results_json["methods"]
    if x["method"] != METHOD_NAME
]

results_json["methods"].append(result_entry)

# ============================================================
# SAVE JSON
# ============================================================
with open(RESULTS_FILE, "w") as f:
    json.dump(results_json, f, indent=4)

print("Saved results to:", RESULTS_FILE)
print(json.dumps(result_entry, indent=4))
print(f"F1 ({in_domain_metrics['f1_average']}): {in_domain_metrics['f1']:.6f}")


In [ ]:
if RUN_CROSS_TASK_TRAINING:
    # ============================================================
    # CURRENT MODE: CROSS-TASK TRAINING / EVALUATION
    # Freeze expert backbone + train only new cross-task classifier
    # ============================================================

    CURRENT_MODE = "cross_task"

    METHOD_NAME = CROSS_TASK_METHOD
    EVAL_TASK = CROSS_TASK
    NUM_CLASSES = CROSS_NUM_CLASSES

    # ------------------------------------------------------------
    # Use frozen expert backbone + new cross-task classifier
    # ------------------------------------------------------------
    active_classifier = cross_classifier

    bert_backbone = bert_backbone.to(device)
    active_classifier = active_classifier.to(device)

    # ------------------------------------------------------------
    # Freeze backbone
    # ------------------------------------------------------------
    bert_backbone.eval()

    for param in bert_backbone.parameters():
        param.requires_grad = False

    # ------------------------------------------------------------
    # Train only cross-task classifier
    # ------------------------------------------------------------
    active_classifier.train()

    for param in active_classifier.parameters():
        param.requires_grad = True

    # ------------------------------------------------------------
    # Optimizer only on cross-task classifier
    # ------------------------------------------------------------
    optimizer = torch.optim.AdamW(
        active_classifier.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.CrossEntropyLoss()

    # ------------------------------------------------------------
    # Scheduler
    # ------------------------------------------------------------
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        min_lr=MIN_LR
    )

    # ------------------------------------------------------------
    # Count trainable parameters
    # ------------------------------------------------------------
    TRAINABLE_PARAMS = (
        sum(p.numel() for p in bert_backbone.parameters() if p.requires_grad)
        + sum(p.numel() for p in active_classifier.parameters() if p.requires_grad)
    )

    print("Current mode:", CURRENT_MODE)
    print("Method name:", METHOD_NAME)
    print("Expert task:", EXPERT_TASK)
    print("Cross task:", EVAL_TASK)
    print("Num classes:", NUM_CLASSES)
    print("Trainable params:", TRAINABLE_PARAMS)
    print("Backbone frozen:", not any(p.requires_grad for p in bert_backbone.parameters()))
    print("Cross classifier trainable:", any(p.requires_grad for p in active_classifier.parameters()))

In [ ]:
if RUN_CROSS_TASK_TRAINING:
    # ============================================================
    # CROSS-TASK MODE
    # Freeze expert backbone + train only new cross-task classifier
    # With early stopping, best model saving, LR scheduler
    # ============================================================

    import json
    import torch
    import torch.nn as nn
    from torch.optim.lr_scheduler import ReduceLROnPlateau

    CURRENT_MODE = "cross_task"

    METHOD_NAME = CROSS_TASK_METHOD
    EVAL_TASK = CROSS_TASK
    NUM_CLASSES = CROSS_NUM_CLASSES

    # ============================================================
    # TRAINING CONTROL PARAMETERS
    # ============================================================
    NUM_EPOCHS = 10

    EARLY_STOPPING_PATIENCE = 4
    EARLY_STOPPING_MIN_DELTA = 1e-4

    LR_THRESHOLD = 1e-4

    # ============================================================
    # OUTPUT PATHS
    # ============================================================
    SAFE_METHOD_NAME = METHOD_NAME.replace("/", "_").replace(" ", "_")

    BEST_MODEL_DIR = MODELS_DIR / f"best_{SAFE_METHOD_NAME}"
    BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

    BEST_CLASSIFIER_PATH = BEST_MODEL_DIR / "classifier.pt"
    BEST_TRAINING_INFO_PATH = BEST_MODEL_DIR / "training_info.json"

    HISTORY_PATH = RESULTS_DIR / f"{SAFE_METHOD_NAME}_training_history.json"
    SUMMARY_PATH = RESULTS_DIR / f"{SAFE_METHOD_NAME}_training_summary.json"

    # ============================================================
    # CREATE NEW CROSS-TASK CLASSIFIER
    # ============================================================
    active_classifier = Classifier(class_number=CROSS_NUM_CLASSES).to(device)

    # ============================================================
    # MOVE BACKBONE TO DEVICE
    # ============================================================
    bert_backbone = bert_backbone.to(device)

    # ============================================================
    # FREEZE BACKBONE
    # ============================================================
    bert_backbone.eval()

    for param in bert_backbone.parameters():
        param.requires_grad = False

    # ============================================================
    # TRAIN ONLY CROSS-TASK CLASSIFIER
    # ============================================================
    active_classifier.train()

    for param in active_classifier.parameters():
        param.requires_grad = True

    # ============================================================
    # LOSS FUNCTION
    # ============================================================
    criterion = nn.CrossEntropyLoss()

    # ============================================================
    # OPTIMIZER
    # Only cross-task classifier parameters are optimized
    # ============================================================
    optimizer = torch.optim.AdamW(
        active_classifier.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # ============================================================
    # SCHEDULER
    # Reduce LR when validation loss stops improving enough
    # ============================================================
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        threshold=LR_THRESHOLD,
        threshold_mode="abs",
        min_lr=MIN_LR
    )

    # ============================================================
    # EARLY STOPPING VARIABLES
    # Best model will be selected using validation F1
    # ============================================================
    best_val_f1 = -float("inf")
    best_epoch = -1
    early_stopping_counter = 0

    training_history = []

    # ============================================================
    # TRAINABLE PARAMETERS
    # ============================================================
    BACKBONE_FROZEN_PARAMS = sum(
        p.numel() for p in bert_backbone.parameters()
        if not p.requires_grad
    )

    CLASSIFIER_TRAINABLE_PARAMS = sum(
        p.numel() for p in active_classifier.parameters()
        if p.requires_grad
    )

    TRAINABLE_PARAMS = CLASSIFIER_TRAINABLE_PARAMS

    FROZEN_PARAMS = BACKBONE_FROZEN_PARAMS

    TOTAL_PARAMS = TRAINABLE_PARAMS + FROZEN_PARAMS

    # ============================================================
    # SANITY CHECKS
    # ============================================================
    print("Current mode:", CURRENT_MODE)
    print("Method name:", METHOD_NAME)
    print("Expert task:", EXPERT_TASK)
    print("Cross task:", EVAL_TASK)
    print("Num classes:", NUM_CLASSES)

    print("Backbone device:", next(bert_backbone.parameters()).device)
    print("Classifier device:", next(active_classifier.parameters()).device)

    print("Learning rate:", LEARNING_RATE)
    print("Weight decay:", WEIGHT_DECAY)
    print("LR factor:", LR_FACTOR)
    print("LR patience:", LR_PATIENCE)
    print("LR threshold:", LR_THRESHOLD)
    print("Min LR:", MIN_LR)

    print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
    print("Early stopping min delta:", EARLY_STOPPING_MIN_DELTA)

    print("Trainable params in classifier:", CLASSIFIER_TRAINABLE_PARAMS)
    print("Trainable params in whole model:", TRAINABLE_PARAMS)
    print("Frozen backbone params:", FROZEN_PARAMS)
    print("Total params:", TOTAL_PARAMS)

    print("Best model dir:", BEST_MODEL_DIR)
    print("History path:", HISTORY_PATH)
    print("Summary path:", SUMMARY_PATH)

    print("\nTrainable layers:")
    for name, param in active_classifier.named_parameters():
        if param.requires_grad:
            print("classifier." + name)

In [ ]:
if RUN_CROSS_TASK_TRAINING:
    # ============================================================
    # PRINT TRAINABLE PARAMETERS
    # ============================================================

    print("Trainable parameters:")

    for name, param in active_classifier.named_parameters():
        if param.requires_grad:
            print("classifier." + name)

In [ ]:
if RUN_CROSS_TASK_TRAINING:
    # ============================================================
    # TRAIN LOOP WITH VALIDATION, SCHEDULER, EARLY STOPPING
    # Frozen backbone caching: the BERT backbone is frozen, so we
    # compute pooled_output once and train only the cross-task classifier.
    # ============================================================

    import json
    import torch

    from tqdm import tqdm
    from torch.utils.data import TensorDataset, DataLoader
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    # ============================================================
    # PARAMETERS
    # ============================================================
    EPOCHS = NUM_EPOCHS

    METHOD_NAME = CROSS_TASK_METHOD
    EVAL_TASK = CROSS_TASK
    MODEL_FILE = CROSS_MODEL_FILE

    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    MODEL_FILE.parent.mkdir(parents=True, exist_ok=True)

    # ============================================================
    # METRICS
    # ============================================================
    def compute_metrics(labels, preds, num_classes):
        average_type = "binary" if num_classes == 2 else "macro"

        return {
            "accuracy": float(accuracy_score(labels, preds)),
            "precision": float(precision_score(labels, preds, average=average_type, zero_division=0)),
            "recall": float(recall_score(labels, preds, average=average_type, zero_division=0)),
            "f1": float(f1_score(labels, preds, average=average_type, zero_division=0)),
        }

    # ============================================================
    # CACHE BACKBONE FEATURES
    # ============================================================
    def cache_backbone_features(backbone, dataloader, device, split_name):
        backbone.eval()

        features = []
        labels_list = []

        with torch.no_grad():
            for batch in tqdm(dataloader, desc=f"Caching {split_name} features"):
                input_ids, attention_mask, token_type_ids, labels = [x.to(device) for x in batch]

                outputs = backbone(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )

                pooled_output = outputs.pooler_output.detach().cpu()

                features.append(pooled_output)
                labels_list.append(labels.detach().cpu())

        features = torch.cat(features, dim=0)
        labels = torch.cat(labels_list, dim=0)

        return TensorDataset(features, labels)


    train_features = cache_backbone_features(
        backbone=bert_backbone,
        dataloader=train_loader_cross,
        device=device,
        split_name=f"{EVAL_TASK}_train"
    )

    val_features = cache_backbone_features(
        backbone=bert_backbone,
        dataloader=val_loader_cross,
        device=device,
        split_name=f"{EVAL_TASK}_val"
    )

    train_feature_loader = DataLoader(train_features, batch_size=BATCH_SIZE, shuffle=True)
    val_feature_loader = DataLoader(val_features, batch_size=BATCH_SIZE, shuffle=False)

    print("Backbone features cached successfully.")

    # ============================================================
    # FORWARD ON CACHED FEATURES
    # ============================================================
    def forward_cached_batch(classifier, batch, criterion, device):
        pooled_output, labels = batch

        pooled_output = pooled_output.to(device)
        labels = labels.to(device)

        logits = classifier(pooled_output)
        loss = criterion(logits, labels)
        preds = torch.argmax(logits, dim=1)

        return loss, preds, labels

    # ============================================================
    # TRAINING FUNCTION
    # ============================================================
    def train_one_epoch(classifier, dataloader, optimizer, criterion, device, num_classes, epoch):
        classifier.train()

        total_loss = 0.0
        total_samples = 0

        all_preds = []
        all_labels = []

        progress_bar = tqdm(
            dataloader,
            desc=f"Epoch {epoch + 1}/{EPOCHS} - Training",
            leave=True
        )

        for batch in progress_bar:
            optimizer.zero_grad(set_to_none=True)

            loss, preds, labels = forward_cached_batch(
                classifier=classifier,
                batch=batch,
                criterion=criterion,
                device=device
            )

            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "lr": f"{optimizer.param_groups[0]['lr']:.1e}"
            })

        metrics = compute_metrics(all_labels, all_preds, num_classes)
        metrics["loss"] = float(total_loss / total_samples)
        metrics["num_samples"] = int(total_samples)

        return metrics

    # ============================================================
    # VALIDATION FUNCTION
    # ============================================================
    def evaluate_one_epoch(classifier, dataloader, criterion, device, num_classes, epoch):
        classifier.eval()

        total_loss = 0.0
        total_samples = 0

        all_preds = []
        all_labels = []

        progress_bar = tqdm(
            dataloader,
            desc=f"Epoch {epoch + 1}/{EPOCHS} - Validation",
            leave=True
        )

        with torch.no_grad():
            for batch in progress_bar:
                loss, preds, labels = forward_cached_batch(
                    classifier=classifier,
                    batch=batch,
                    criterion=criterion,
                    device=device
                )

                batch_size = labels.size(0)

                total_loss += loss.item() * batch_size
                total_samples += batch_size

                all_preds.extend(preds.detach().cpu().numpy())
                all_labels.extend(labels.detach().cpu().numpy())

                progress_bar.set_postfix({
                    "loss": f"{loss.item():.4f}"
                })

        metrics = compute_metrics(all_labels, all_preds, num_classes)
        metrics["loss"] = float(total_loss / total_samples)
        metrics["num_samples"] = int(total_samples)

        return metrics

    # ============================================================
    # TRAIN LOOP
    # ============================================================
    train_history = []

    best_val_f1 = -float("inf")
    best_epoch = -1
    early_stopping_counter = 0

    for epoch in range(EPOCHS):

        print("\n" + "=" * 80)
        print(f"Epoch {epoch + 1}/{EPOCHS}")
        print("=" * 80)

        train_metrics = train_one_epoch(
            classifier=active_classifier,
            dataloader=train_feature_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            num_classes=CROSS_NUM_CLASSES,
            epoch=epoch
        )

        val_metrics = evaluate_one_epoch(
            classifier=active_classifier,
            dataloader=val_feature_loader,
            criterion=criterion,
            device=device,
            num_classes=CROSS_NUM_CLASSES,
            epoch=epoch
        )

        scheduler.step(val_metrics["loss"])

        current_lr = float(optimizer.param_groups[0]["lr"])

        is_best = val_metrics["f1"] > best_val_f1 + EARLY_STOPPING_MIN_DELTA

        if is_best:
            best_val_f1 = val_metrics["f1"]
            best_epoch = epoch + 1
            early_stopping_counter = 0

            torch.save(active_classifier.state_dict(), MODEL_FILE)
            torch.save(active_classifier.state_dict(), BEST_CLASSIFIER_PATH)

        else:
            early_stopping_counter += 1

        epoch_record = {
            "epoch": epoch + 1,
            "train": train_metrics,
            "val": val_metrics,
            "learning_rate": current_lr,
            "is_best": is_best,
            "early_stopping_counter": early_stopping_counter,
        }

        train_history.append(epoch_record)

        with open(HISTORY_PATH, "w") as f:
            json.dump(train_history, f, indent=4)

        print(
            f"train_f1={train_metrics['f1']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"lr={current_lr:.8f} | "
            f"best={is_best}"
        )

        if early_stopping_counter >= EARLY_STOPPING_PATIENCE:
            print("\nEarly stopping triggered.")
            break

    # ============================================================
    # FINAL SUMMARY
    # ============================================================
    best_record = train_history[best_epoch - 1]

    summary = {
        "method": METHOD_NAME,
        "mode": CURRENT_MODE,
        "expert_task": EXPERT_TASK,
        "eval_task": EVAL_TASK,
        "num_classes": int(CROSS_NUM_CLASSES),
        "trainable_params": int(TRAINABLE_PARAMS),
        "classifier_trainable_params": int(CLASSIFIER_TRAINABLE_PARAMS),
        "frozen_params": int(FROZEN_PARAMS),
        "best_epoch": int(best_epoch),
        "best_val_f1": float(best_val_f1),
        "best_record": best_record,
        "epochs_completed": int(len(train_history)),
        "learning_rate": float(LEARNING_RATE),
        "weight_decay": float(WEIGHT_DECAY),
        "early_stopping_patience": int(EARLY_STOPPING_PATIENCE),
        "early_stopping_min_delta": float(EARLY_STOPPING_MIN_DELTA),
        "backbone_caching": True,
        "backbone_frozen": True,
        "backbone_path": str(EXPERT_BACKBONE_DIR),
        "classifier_path": str(BEST_CLASSIFIER_PATH),
        "model_path": str(MODEL_FILE),
    }

    with open(SUMMARY_PATH, "w") as f:
        json.dump(summary, f, indent=4)

    with open(BEST_TRAINING_INFO_PATH, "w") as f:
        json.dump(summary, f, indent=4)

    # ============================================================
    # UPDATE MAIN RESULTS JSON
    # ============================================================
    result_entry = {
        "method": METHOD_NAME,
        "mode": CURRENT_MODE,
        "expert_task": EXPERT_TASK,
        "eval_task": EVAL_TASK,
        "trainable_params": int(TRAINABLE_PARAMS),
        "results": {
            EVAL_TASK: {
                "train_history": train_history,
                "best_summary": summary,
            }
        },
        "backbone_path": str(EXPERT_BACKBONE_DIR),
        "classifier_path": str(BEST_CLASSIFIER_PATH),
        "model_path": str(MODEL_FILE)
    }

    if RESULTS_FILE.exists():
        with open(RESULTS_FILE, "r") as f:
            results_json = json.load(f)
    else:
        results_json = {"methods": []}

    results_json["methods"] = [
        x for x in results_json["methods"]
        if x["method"] != METHOD_NAME
    ]

    results_json["methods"].append(result_entry)

    with open(RESULTS_FILE, "w") as f:
        json.dump(results_json, f, indent=4)

    # ============================================================
    # FINAL PRINT ONLY
    # ============================================================
    print("\nTraining completed.")
    print(f"Epochs completed: {len(train_history)}")
    print(f"Best epoch: {best_epoch}")
    print(f"Best validation F1: {best_val_f1:.4f}")
    print(f"Best validation accuracy: {best_record['val']['accuracy']:.4f}")
    print("Classifier saved to:", MODEL_FILE)
    print("History saved to:", HISTORY_PATH)
    print("Summary saved to:", SUMMARY_PATH)
    print("Results saved to:", RESULTS_FILE)

In [ ]:
if RUN_CROSS_TASK_TRAINING:
    # ============================================================
    # CROSS-TASK TEST EVALUATION
    # Frozen backbone caching: compute pooled_output once, then
    # evaluate the trained cross-task classifier on cached features.
    # ============================================================

    import json
    import torch

    from tqdm import tqdm
    from torch.utils.data import TensorDataset, DataLoader
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

    # ============================================================
    # PARAMETERS
    # ============================================================
    METHOD_NAME = CROSS_TASK_METHOD
    EVAL_TASK = CROSS_TASK
    MODEL_FILE = CROSS_MODEL_FILE

    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    MODEL_FILE.parent.mkdir(parents=True, exist_ok=True)

    # ============================================================
    # LOAD BEST CROSS-TASK CLASSIFIER IF AVAILABLE
    # ============================================================
    if MODEL_FILE.exists():
        active_classifier.load_state_dict(torch.load(MODEL_FILE, map_location=device))
        print("Loaded best cross-task classifier from:", MODEL_FILE)
    else:
        print("Best classifier file not found. Evaluating current classifier in memory.")

    bert_backbone = bert_backbone.to(device)
    active_classifier = active_classifier.to(device)

    bert_backbone.eval()
    active_classifier.eval()

    # ============================================================
    # CACHE TEST FEATURES
    # ============================================================
    def cache_backbone_features(backbone, dataloader, device, split_name, inference_cost):
        backbone.eval()

        features = []
        labels_list = []

        with torch.no_grad():
            for batch in tqdm(dataloader, desc=f"Caching {split_name} features"):
                input_ids, attention_mask, token_type_ids, labels = [x.to(device) for x in batch]

                with InferenceFlopCounter() as backbone_counter:
                    outputs = backbone(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids
                    )
                add_inference_cost(inference_cost, backbone_counter, "backbone", input_ids)

                pooled_output = outputs.pooler_output.detach().cpu()

                features.append(pooled_output)
                labels_list.append(labels.detach().cpu())

        features = torch.cat(features, dim=0)
        labels = torch.cat(labels_list, dim=0)

        return TensorDataset(features, labels)


    inference_cost_cross = new_inference_cost()

    test_features_cross = cache_backbone_features(
        backbone=bert_backbone,
        dataloader=test_loader_cross,
        device=device,
        split_name=f"{EVAL_TASK}_test",
        inference_cost=inference_cost_cross
    )

    test_feature_loader_cross = DataLoader(
        test_features_cross,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # ============================================================
    # EVALUATION FUNCTION ON CACHED FEATURES
    # ============================================================
    def evaluate_cross_task_cached(classifier, dataloader, criterion, device, task_name, num_classes, inference_cost):
        classifier.eval()

        total_loss = 0.0
        total_samples = 0

        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(dataloader, desc=f"Evaluating on {task_name.upper()} test"):
                pooled_output, labels = batch

                pooled_output = pooled_output.to(device)
                labels = labels.to(device)

                with InferenceFlopCounter() as classifier_counter:
                    logits = classifier(pooled_output)
                add_inference_cost(inference_cost, classifier_counter, "classifier")
                loss = criterion(logits, labels)

                preds = torch.argmax(logits, dim=1)
                batch_size = labels.size(0)

                total_loss += loss.item() * batch_size
                total_samples += batch_size

                all_preds.extend(preds.detach().cpu().numpy())
                all_labels.extend(labels.detach().cpu().numpy())

        average_type = "binary" if num_classes == 2 else "macro"

        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels,
            all_preds,
            average=average_type,
            zero_division=0
        )

        return {
            "loss": float(total_loss / total_samples),
            "accuracy": float(accuracy_score(all_labels, all_preds)),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "confusion_matrix": confusion_matrix(
                all_labels,
                all_preds,
                labels=list(range(num_classes))
            ).tolist(),
            "num_samples": int(total_samples),
            "backbone_caching": True,
            "inference_cost": finish_inference_cost(inference_cost, total_samples)
        }

    # ============================================================
    # RUN EVALUATION
    # ============================================================
    test_metrics_cross = evaluate_cross_task_cached(
        classifier=active_classifier,
        dataloader=test_feature_loader_cross,
        criterion=criterion,
        device=device,
        task_name=EVAL_TASK,
        num_classes=CROSS_NUM_CLASSES,
        inference_cost=inference_cost_cross
    )

    # ============================================================
    # UPDATE JSON
    # ============================================================
    if RESULTS_FILE.exists():
        with open(RESULTS_FILE, "r") as f:
            results_json = json.load(f)
    else:
        results_json = {"methods": []}

    method_found = False

    for entry in results_json["methods"]:
        if entry["method"] == METHOD_NAME:
            entry.setdefault("results", {})
            entry["results"].setdefault(EVAL_TASK, {})

            entry["results"][EVAL_TASK]["test_metrics"] = test_metrics_cross
            entry["trainable_params"] = int(TRAINABLE_PARAMS)
            entry["backbone_path"] = str(EXPERT_BACKBONE_DIR)
            entry["classifier_path"] = str(MODEL_FILE)
            entry["model_path"] = str(MODEL_FILE)

            method_found = True
            break

    if not method_found:
        results_json["methods"].append({
            "method": METHOD_NAME,
            "mode": CURRENT_MODE,
            "expert_task": EXPERT_TASK,
            "eval_task": EVAL_TASK,
            "trainable_params": int(TRAINABLE_PARAMS),
            "results": {
                EVAL_TASK: {
                    "test_metrics": test_metrics_cross
                }
            },
            "backbone_path": str(EXPERT_BACKBONE_DIR),
            "classifier_path": str(MODEL_FILE),
            "model_path": str(MODEL_FILE)
        })

    with open(RESULTS_FILE, "w") as f:
        json.dump(results_json, f, indent=4)

    # ============================================================
    # PRINT SHORT SUMMARY
    # ============================================================
    print("\nTEST RESULTS")
    print("=" * 80)
    print("Method:", METHOD_NAME)
    print("Expert task:", EXPERT_TASK)
    print("Test task:", EVAL_TASK)
    print(f"Loss:      {test_metrics_cross['loss']:.4f}")
    print(f"Accuracy:  {test_metrics_cross['accuracy']:.4f}")
    print(f"Precision: {test_metrics_cross['precision']:.4f}")
    print(f"Recall:    {test_metrics_cross['recall']:.4f}")
    print(f"F1:        {test_metrics_cross['f1']:.4f}")
    print("Results saved to:", RESULTS_FILE)

In [ ]:
import json
import numpy as np
from pathlib import Path

# ============================================================
# PARAMETERS
# ============================================================
RESULTS_FILE = Path(RESULTS_FILE)

# ============================================================
# PRETTY PRINT UTILITIES
# ============================================================
def print_separator(char="=", n=80):
    print(char * n)


def print_title(title):
    print_separator("=")
    print(title)
    print_separator("=")


def print_subtitle(subtitle):
    print(f"\n{subtitle}")
    print_separator("-", len(subtitle))


def print_metric(name, value, decimals=4):
    if isinstance(value, float):
        print(f"{name:<22}: {value:.{decimals}f}")
    else:
        print(f"{name:<22}: {value}")


def print_confusion_matrix(cm):
    cm = np.array(cm)

    if cm.ndim != 2:
        print("Confusion matrix not available in valid format.")
        return

    n_rows, n_cols = cm.shape
    cell_width = max(8, max(len(str(x)) for x in cm.flatten()) + 2)

    print("\nConfusion Matrix")
    print_separator("-", 18)

    header = " " * 12 + "".join([f"Pred {j:<{cell_width-5}}" for j in range(n_cols)])
    print(header)

    for i in range(n_rows):
        row = f"True {i:<6}" + "".join([f"{cm[i, j]:<{cell_width}}" for j in range(n_cols)])
        print(row)


def print_train_history(train_history):
    if not train_history:
        print("No training history available.")
        return

    print("\nTraining History")
    print_separator("-", 16)

    header = (
        f"{'Epoch':<8}"
        f"{'Train Loss':<14}{'Train Acc':<12}{'Train F1':<12}"
        f"{'Val Loss':<14}{'Val Acc':<12}{'Val F1':<12}"
        f"{'LR':<12}{'Best':<8}"
    )

    print(header)
    print_separator("-", len(header))

    for i, epoch_record in enumerate(train_history, start=1):
        epoch_num = epoch_record.get("epoch", i)

        train_metrics = epoch_record.get("train", {})
        val_metrics = epoch_record.get("val", {})

        lr = epoch_record.get("learning_rate", 0)
        is_best = epoch_record.get("is_best", False)

        print(
            f"{epoch_num:<8}"
            f"{train_metrics.get('loss', 0):<14.4f}"
            f"{train_metrics.get('accuracy', 0):<12.4f}"
            f"{train_metrics.get('f1', 0):<12.4f}"
            f"{val_metrics.get('loss', 0):<14.4f}"
            f"{val_metrics.get('accuracy', 0):<12.4f}"
            f"{val_metrics.get('f1', 0):<12.4f}"
            f"{lr:<12.2e}"
            f"{str(is_best):<8}"
        )


def print_best_summary(task_results):
    best_summary = task_results.get("best_summary", None)

    if best_summary is None:
        return

    print("\nBest Summary")
    print_separator("-", 12)

    print_metric("Best epoch", best_summary.get("best_epoch", "N/A"))
    print_metric("Best val F1", best_summary.get("best_val_f1", "N/A"))
    print_metric("Epochs completed", best_summary.get("epochs_completed", "N/A"))
    print_metric("Learning rate", best_summary.get("learning_rate", "N/A"))
    print_metric("Weight decay", best_summary.get("weight_decay", "N/A"))
    print_metric("Backbone caching", best_summary.get("backbone_caching", "N/A"))
    print_metric("Backbone frozen", best_summary.get("backbone_frozen", "N/A"))


def print_test_metrics(test_metrics):
    if not test_metrics:
        print("No test metrics available.")
        return

    print("\nTest Metrics")
    print_separator("-", 12)

    print_metric("Loss", test_metrics.get("loss", "N/A"))
    print_metric("Accuracy", test_metrics.get("accuracy", "N/A"))
    print_metric("Precision", test_metrics.get("precision", "N/A"))
    print_metric("Recall", test_metrics.get("recall", "N/A"))
    print_metric("F1-score", test_metrics.get("f1", "N/A"))
    print_metric("Num samples", test_metrics.get("num_samples", "N/A"))
    print_metric("Backbone caching", test_metrics.get("backbone_caching", "N/A"))

    cost = test_metrics.get("inference_cost")
    if cost is not None:
        coverage = cost["coverage"]
        print_metric("FLOPs coverage", coverage["status"])
        if cost["total_flops"] is not None:
            print_metric("Inference FLOPs", f"{cost['total_flops']:,}")
            print_metric("FLOPs per example", f"{cost['flops_per_example']:,.2f}")
        else:
            print_metric("Inference FLOPs", "Incomplete count; see JSON coverage")
            print_metric("Uncovered operators", ", ".join(coverage["uncovered_operators"]))
        print_metric("FLOPs MAX_LEN", cost["input"]["max_len"])
        print_metric("FLOPs batch size", cost["input"]["batch_size"])

    if "confusion_matrix" in test_metrics:
        print_confusion_matrix(test_metrics["confusion_matrix"])


def extract_test_metrics(task_results):
    if "test_metrics" in task_results:
        return task_results["test_metrics"]

    if "test" in task_results:
        return task_results["test"]

    return None


# ============================================================
# LOAD JSON
# ============================================================
if not RESULTS_FILE.exists():
    raise FileNotFoundError(f"Results file not found: {RESULTS_FILE}")

with open(RESULTS_FILE, "r") as f:
    results_json = json.load(f)

# ============================================================
# PRINT RESULTS
# ============================================================
methods = results_json.get("methods", [])

if not methods:
    print("No methods found in the JSON file.")

else:
    print_title("EXPERIMENT RESULTS")
    print("Results file:", RESULTS_FILE)

    for method_entry in methods:
        method_name = method_entry.get("method", "Unknown method")
        mode = method_entry.get("mode", "N/A")
        expert_task = method_entry.get("expert_task", "N/A")
        eval_task = method_entry.get("eval_task", "N/A")
        trainable_params = method_entry.get("trainable_params", "N/A")

        backbone_path = method_entry.get("backbone_path", "N/A")
        classifier_path = method_entry.get("classifier_path", "N/A")
        model_path = method_entry.get("model_path", "N/A")

        results = method_entry.get("results", {})

        print_subtitle(f"Method: {method_name}")

        print_metric("Mode", mode)
        print_metric("Expert task", expert_task)
        print_metric("Eval task", eval_task)
        print_metric("Trainable params", trainable_params)
        print_metric("Backbone path", backbone_path)
        print_metric("Classifier path", classifier_path)
        print_metric("Model path", model_path)

        for task_name, task_results in results.items():
            print_subtitle(f"Task: {task_name}")

            if "train_history" in task_results:
                print_train_history(task_results["train_history"])

            print_best_summary(task_results)

            test_metrics = extract_test_metrics(task_results)

            if test_metrics is not None:
                print_test_metrics(test_metrics)
            else:
                print("No test metrics found.")